In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
from PIL import Image
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms as T
import timm
from safetensors.torch import load_file

warnings.filterwarnings("ignore")
pd.set_option('mode.chained_assignment', None)

# ================== CONFIG ==================
SEED = 42
DATA_DIR = '/kaggle/input/csiro-biomass'
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
LOSS_WEIGHTS = torch.tensor([1.0, 3.0, 2.0, 1.0, 2.5], dtype=torch.float32)
EVAL_WEIGHTS = torch.tensor([0.25, 0.25, 0.15, 0.20, 0.15], dtype=torch.float32)
IMG_SIZE = 512
BS = 2
N_FOLDS = 4
TOTAL_EPOCHS = 30
LR_MAX = 5e-5 # Slightly increased for the new backbone
DINOV2_PATH = "/kaggle/input/dinov2-large-update-2/dinov2_large/dinov2_vitl14_pretrain.pth"
SIGLIP_PATH = "/kaggle/input/siglip-large-model/Siglip_large_Update/model.safetensors"
EFFNET_PATH = "/kaggle/input/efficientnet-b5-update/model.safetensors"

# ─── Recommended tuning values ───
CONSISTENCY_WEIGHT = 1.0          
SMOOTH_L1_BETA = 0.1             

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================== DATA PROCESSING ==================
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")\
             .pivot_table(index='image_path', columns='target_name', values='target', aggfunc='first')\
             .reset_index()
train_df = train_df[['image_path'] + TARGETS].dropna().reset_index(drop=True)
train_targets = np.log1p(train_df[TARGETS].values)
t_mean = torch.tensor(train_targets.mean(axis=0)).to(DEVICE)
t_std  = torch.tensor(train_targets.std(axis=0) + 1e-6).to(DEVICE)
train_norm = (train_targets - t_mean.cpu().numpy()) / t_std.cpu().numpy()
test_files = [os.path.basename(p) for p in pd.read_csv(f"{DATA_DIR}/test.csv")['image_path'].unique()]

# ================== MODEL ==================
class BiomassXpressModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. DINOv2 Large
        self.dinov2 = timm.create_model('vit_large_patch14_dinov2.lvd142m', 
                                       pretrained=False, num_classes=0, img_size=IMG_SIZE)
        d_state = torch.load(DINOV2_PATH, map_location='cpu')
        # Pos embed interpolation logic
        pos_ckpt = d_state['pos_embed']
        n_patches = self.dinov2.patch_embed.num_patches
        n_extra = self.dinov2.pos_embed.shape[-2] - n_patches
        extra, patches = pos_ckpt[:, :n_extra], pos_ckpt[:, n_extra:]
        gs = int(np.sqrt(patches.shape[1]))
        patches = F.interpolate(
            patches.reshape(1, gs, gs, -1).permute(0,3,1,2),
            size=(int(np.sqrt(n_patches)), int(np.sqrt(n_patches))),
            mode='bicubic', align_corners=False
        )
        d_state['pos_embed'] = torch.cat((extra, patches.permute(0,2,3,1).reshape(1, n_patches, -1)), dim=1)
        self.dinov2.load_state_dict(d_state, strict=False)

        # 2. SigLIP Large
        self.siglip = timm.create_model('vit_large_patch16_siglip_512', 
                                       pretrained=False, num_classes=0)
        s_state = load_file(SIGLIP_PATH)
        self.siglip.load_state_dict(s_state, strict=False)

        # 3. EfficientNet-B5 (CNN Backbone)
        self.effnet = timm.create_model('efficientnet_b5', pretrained=False, num_classes=0)
        e_state = load_file(EFFNET_PATH)
        self.effnet.load_state_dict(e_state, strict=False)

        # Combined Concatenation: 1024 (DINO) + 1024 (SigLIP) + 2048 (EffNet-B5) = 4096
        self.head_base = nn.Sequential(
            nn.Linear(4096, 1024),
            nn.LayerNorm(1024),
            nn.SiLU() # Switching GELU to SiLU for CNN compatibility
        )
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * (i+1)) for i in range(5)])
        self.regressor = nn.Linear(1024, 5)

    def forward(self, x):
        f1 = self.dinov2(x)
        f2 = self.siglip(x)
        f3 = self.effnet(x)
        feat = self.head_base(torch.cat([f1, f2, f3], dim=1))
        # Multi-sample dropout for regression stability
        return torch.mean(torch.stack([self.regressor(drop(feat)) for drop in self.dropouts]), dim=0)

# ================== IMPROVED LOSS ==================
def biomass_precision_loss(pred, target):
    # Weighted Smooth L1
    base_loss = F.smooth_l1_loss(pred, target, reduction='none', beta=SMOOTH_L1_BETA)
    weighted_loss = (base_loss * LOSS_WEIGHTS.to(DEVICE)).sum(1).mean()

    # Physical consistency: sum of parts = total
    p_orig = torch.expm1(pred * t_std + t_mean)
    consist_loss = F.mse_loss(
        torch.log1p(p_orig[:,0] + p_orig[:,1] + p_orig[:,2]),
        torch.log1p(p_orig[:,4])
    )
    
    return weighted_loss + (CONSISTENCY_WEIGHT * consist_loss)

# ================== TRAINING UTILS ==================
def train_fold(fold, tr_idx, val_idx):
    train_dl = DataLoader(BiomassDataset(train_df.iloc[tr_idx], train_norm[tr_idx], train_aug),
                          batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(BiomassDataset(train_df.iloc[val_idx], train_norm[val_idx], val_aug),
                          batch_size=BS, num_workers=2, pin_memory=True)

    model = BiomassXpressModel().to(DEVICE)
    
    optimizer = torch.optim.AdamW([
        {'params': model.dinov2.parameters(),   'lr': LR_MAX / 40},
        {'params': model.siglip.parameters(),   'lr': LR_MAX / 40},
        {'params': model.effnet.parameters(),   'lr': LR_MAX / 20},
        {'params': model.head_base.parameters(), 'lr': LR_MAX},
        {'params': model.regressor.parameters(), 'lr': LR_MAX},
    ], weight_decay=0.01)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    scaler = GradScaler()

    best_r2 = -1
    for epoch in range(TOTAL_EPOCHS):
        model.train()
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with autocast():
                loss = biomass_precision_loss(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        scheduler.step()

        # Validation
        model.eval()
        preds, targs = [], []
        with torch.no_grad():
            for x, y in val_dl:
                preds.append(model(x.to(DEVICE)).cpu().numpy())
                targs.append(y.numpy())

        preds = np.concatenate(preds)
        targs = np.concatenate(targs)
        preds_expm1 = np.expm1(preds * t_std.cpu().numpy() + t_mean.cpu().numpy())
        targs_expm1 = np.expm1(targs * t_std.cpu().numpy() + t_mean.cpu().numpy())

        r2 = np.sum([r2_score(targs_expm1[:,i], preds_expm1[:,i]) * EVAL_WEIGHTS[i].item() 
                     for i in range(5)])

        if r2 > best_r2:
            best_r2 = r2
            torch.save(model.state_dict(), f"best_fold_{fold}.pth")
            print(f"Fold {fold} | Ep {epoch} | R2: {r2:.4f} *")

    del model, optimizer, train_dl, val_dl
    gc.collect()
    torch.cuda.empty_cache()
    return best_r2

# ================== AUGMENTATIONS & DATASET ==================
train_aug = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), # Less aggressive crop
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05)), # New geometric transform
    T.ColorJitter(0.1, 0.1, 0.1),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)])

val_aug = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)])

class BiomassDataset(Dataset):
    def __init__(self, df, targets, transform):
        self.paths = df['image_path'].values
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        p = os.path.join(DATA_DIR, 'train' if self.targets is not None else 'test',
                         os.path.basename(self.paths[i]))
        img = Image.open(p).convert('RGB')
        x = self.transform(img)
        
        if self.targets is not None:
            return x, torch.tensor(self.targets[i], dtype=torch.float32)
        return x, 0

# ================== TRAINING LOOP ==================
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for f, (ti, vi) in enumerate(kf.split(train_df)):
    print(f"\n===== Fold {f} =====")
    train_fold(f, ti, vi)

# ================== INFERENCE + TTA ==================
test_ds = BiomassDataset(pd.DataFrame({'image_path': test_files}), None, val_aug)
test_dl = DataLoader(test_ds, batch_size=BS*2, shuffle=False, num_workers=2)
final_preds = np.zeros((len(test_files), 5))

for f in range(N_FOLDS):
    print(f"Inference fold {f}...")
    m = BiomassXpressModel().to(DEVICE)
    m.load_state_dict(torch.load(f"best_fold_{f}.pth"))
    m.eval()

    p_fold = []
    with torch.no_grad():
        for x, _ in test_dl:
            x = x.to(DEVICE)
            # 4-way TTA: Normal, Horizontal, Vertical, Both
            p1 = m(x)
            p2 = m(torch.flip(x, [3]))
            p3 = m(torch.flip(x, [2]))
            p4 = m(torch.flip(x, [2, 3]))
            p_fold.append(((p1 + p2 + p3 + p4) / 4).cpu().numpy())

    final_preds += (np.concatenate(p_fold) / N_FOLDS)

# Post-processing
final_preds = np.clip(np.expm1(final_preds * t_std.cpu().numpy() + t_mean.cpu().numpy()), 0, None)

# ================== SUBMISSION ==================
sub_df = pd.DataFrame(final_preds, columns=TARGETS)
sub_df['image_path'] = test_files
final_sub = pd.read_csv(f"{DATA_DIR}/test.csv").copy()
final_sub['image_path'] = final_sub['image_path'].apply(os.path.basename)

final_sub = final_sub.merge(
    sub_df.melt(id_vars='image_path', var_name='target_name', value_name='target'),
    on=['image_path', 'target_name'])

final_sub[['sample_id', 'target']].sort_values('sample_id').to_csv('submission.csv', index=False)
print("\nFinal R2 Focused submission created: submission.csv")

/kaggle/input/siglip-base-512/config.json
/kaggle/input/siglip-base-512/README.md
/kaggle/input/siglip-base-512/open_clip_pytorch_model.bin
/kaggle/input/siglip-base-512/gitattributes
/kaggle/input/siglip-base-512/open_clip_config.json
/kaggle/input/siglip-base-512/open_clip_model.safetensors
/kaggle/input/dinov2-base/config.json
/kaggle/input/dinov2-base/dinov2_vitb14_pretrain.pth
/kaggle/input/dinov2-base/README.md
/kaggle/input/dinov2-base/gitattributes
/kaggle/input/convnext-model-base-384/config.json
/kaggle/input/convnext-model-base-384/gitattributes (1)
/kaggle/input/convnext-model-base-384/README (1).md
/kaggle/input/convnext-model-base-384/preprocessor_config.json
/kaggle/input/convnext-model-base-384/pytorch_model.bin
/kaggle/input/convnext-model-base-384/model.safetensors
/kaggle/input/efficientnet-b5-update/config.json
/kaggle/input/efficientnet-b5-update/README.md
/kaggle/input/efficientnet-b5-update/gitattributes
/kaggle/input/efficientnet-b5-update/pytorch_model.bin
/kag

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 


===== Fold 0 =====
Fold 0 | Ep 0 | R2: 0.4804 *
Fold 0 | Ep 1 | R2: 0.4873 *
Fold 0 | Ep 2 | R2: 0.5276 *
Fold 0 | Ep 4 | R2: 0.5815 *
Fold 0 | Ep 5 | R2: 0.5976 *
Fold 0 | Ep 7 | R2: 0.7067 *

===== Fold 1 =====
Fold 1 | Ep 0 | R2: 0.2072 *
Fold 1 | Ep 1 | R2: 0.4493 *
Fold 1 | Ep 2 | R2: 0.5543 *
Fold 1 | Ep 4 | R2: 0.5827 *
Fold 1 | Ep 5 | R2: 0.5886 *
Fold 1 | Ep 8 | R2: 0.5956 *
Fold 1 | Ep 10 | R2: 0.6056 *
Fold 1 | Ep 11 | R2: 0.6237 *
Fold 1 | Ep 13 | R2: 0.6335 *
Fold 1 | Ep 16 | R2: 0.6459 *
Fold 1 | Ep 23 | R2: 0.6643 *
Fold 1 | Ep 29 | R2: 0.6728 *

===== Fold 2 =====
Fold 2 | Ep 0 | R2: 0.4924 *
Fold 2 | Ep 1 | R2: 0.5479 *
Fold 2 | Ep 2 | R2: 0.5620 *
Fold 2 | Ep 3 | R2: 0.5953 *
Fold 2 | Ep 4 | R2: 0.6136 *
Fold 2 | Ep 6 | R2: 0.6258 *
Fold 2 | Ep 8 | R2: 0.6260 *
Fold 2 | Ep 9 | R2: 0.6732 *
Fold 2 | Ep 10 | R2: 0.6825 *
Fold 2 | Ep 11 | R2: 0.6910 *
Fold 2 | Ep 12 | R2: 0.7084 *
Fold 2 | Ep 23 | R2: 0.7315 *

===== Fold 3 =====
Fold 3 | Ep 0 | R2: 0.2319 *
Fold 3 | Ep